# Aula 5 • Aprendizado Supervisionado I
## Desafio passo a passo: primeiro modelo de produção de veículos

**Problema:** como usar o histórico observado para estimar a produção mensal futura no Brasil?

### Como trabalhar
1. Leia o checkpoint.
2. Preveja o resultado antes de executar.
3. Complete somente o trecho marcado como `COMPLETE`.
4. Execute e verifique tipos, dimensões e datas.
5. Escreva uma interpretação em Markdown.

> A entrega não é apenas o código. Cada gráfico e cada métrica precisam de uma explicação.


## Mapa do desafio

| Checkpoint | Produto |
|---|---|
| 1 | Delimitação correta do problema |
| 2 | Três hipóteses verificáveis |
| 3 | Diagnóstico de ausências e zeros |
| 4 | Três gráficos com interpretação |
| 5 | Features temporais sem vazamento |
| 6 | Investigação contextual de extremos |
| 7 | Treino e teste cronológicos |
| 8 | Baseline ingênuo |
| 9 | Pipeline de Regressão Linear |
| 10 | MAE, MSE, R² e conclusão |

**Regra:** não consulte o teste para escolher features, imputação ou parâmetros.


## 0. Preparar o ambiente

A seed fixa ajuda a reproduzir etapas aleatórias. Em séries temporais, ela **não substitui a divisão cronológica**.

In [ ]:
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import (
    accuracy_score, confusion_matrix,
    mean_absolute_error, mean_squared_error, r2_score
)
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

SEED = 42
np.random.seed(SEED)
sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 25)
warnings.filterwarnings("ignore", category=FutureWarning)
print("Ambiente pronto! Seed:", SEED)

## 1. Carregar a planilha

Envie o arquivo Excel para o Colab quando solicitado. A aba principal é `Dados`; `Sheet1` contém apenas um recorte e não deve ser utilizada.

In [ ]:
ARQUIVO = "PRO_IHS_Producao_Mensal_Veiculos_Motores.xlsx"

if not os.path.exists(ARQUIVO):
    try:
        from google.colab import files
        print("Selecione a planilha fornecida pela professora.")
        arquivos_enviados = files.upload()
        if arquivos_enviados:
            ARQUIVO = next(iter(arquivos_enviados))
    except ImportError:
        raise FileNotFoundError(
            "Coloque a planilha na mesma pasta do notebook ou envie-a pelo Colab."
        )

dados = pd.read_excel(ARQUIVO, sheet_name="Dados")
glossario = pd.read_excel(ARQUIVO, sheet_name="Glossário")

print("Dimensão da aba Dados:", dados.shape)
print("Abas: Dados e Glossário carregadas.")
display(dados.head(3))

## 2. Delimitar o problema

**Responda antes de executar:**

- A variável representa vendas ou produção?
- Qual é o país investigado?
- Qual é o último mês realmente observado?
- Por que a linha de totalização não deve entrar como veículo?

In [ ]:
coluna_pais = "VP: Country/Territory"
coluna_marca = "VP: Production Brand"
coluna_segmento = "Market Segment"
coluna_tipo = "VP: Platform Subgroup Design Type"

brasil = dados[
    dados[coluna_pais].astype(str).str.strip().str.lower().isin(["brazil", "brasil"])
].copy()

colunas_mensais = [
    coluna for coluna in brasil.columns
    if pd.notna(pd.to_datetime(coluna, format="%b %Y", errors="coerce"))
]

print("Registros brasileiros:", len(brasil))
print("Meses encontrados:", len(colunas_mensais))
print("Período:", colunas_mensais[0], "até", colunas_mensais[-1])
print("A linha agregada não foi incluída porque não possui país.")

### DESAFIO 1 — Registrar a delimitação

Complete a resposta:

> A base registra __________ no __________, entre __________ e __________. O primeiro mês futuro que pode ser estimado é __________.

In [ ]:
# Escreva a delimitação da análise em suas palavras.
delimitacao = "COMPLETE: fenômeno, país, período e primeiro mês de previsão."
print(delimitacao)

## 3. Formular três hipóteses

Uma hipótese deve relacionar uma variável ao problema e indicar uma justificativa.

**Modelo:** espera-se que `[variável]` esteja associada a `[resultado]`, porque `[justificativa]`. A relação será investigada por `[análise ou gráfico]`.

In [ ]:
hipoteses = {
    "H1": "COMPLETE: relacione mês e produção.",
    "H2": "COMPLETE: relacione marca e produção.",
    "H3": "COMPLETE: relacione histórico produtivo e previsão."
}

for codigo, texto in hipoteses.items():
    print(f"{codigo}: {texto}")

## 4. Explorar tipos, ausências e estatísticas

Identifique as colunas categóricas, as colunas mensais numéricas e a natureza dos valores ausentes. Atenção: zero de produção não é automaticamente um erro.

In [ ]:
print("Tipos das colunas descritivas:")
print(brasil.iloc[:, :13].dtypes)

print("\nAusências nas colunas categóricas:")
print(brasil.iloc[:, :13].isna().sum())

print("\nResumo das primeiras colunas mensais:")
display(brasil[colunas_mensais[:6]].describe().round(2))

print("\nRegistros com produção total zero:",
      int((brasil[colunas_mensais].sum(axis=1) == 0).sum()))

### DESAFIO 2 — Interpretar o valor ausente

A coluna de tipo de plataforma possui valores ausentes. Verifique se eles aparecem principalmente em determinado segmento e proponha um tratamento coerente.

In [ ]:
# Complete a investigação com groupby, value_counts ou uma tabela cruzada.
# Dica:
# pd.crosstab(brasil[coluna_segmento], brasil[coluna_tipo].isna())

print("Descreva por que não devemos substituir automaticamente essas ausências por ICE.")

## 5. Transformar a base para o formato longo

Na planilha original, cada mês está em uma coluna. Para investigar o tempo, vamos criar uma coluna `data` e uma coluna `producao`.

In [ ]:
categoricas = [
    coluna_segmento,
    coluna_pais,
    "VP: Manufacturer",
    coluna_marca,
    "VP: Production Nameplate",
    coluna_tipo
]

base_longa = brasil.melt(
    id_vars=categoricas,
    value_vars=colunas_mensais,
    var_name="mes_referencia",
    value_name="producao"
)

base_longa["data"] = pd.to_datetime(
    base_longa["mes_referencia"],
    format="%b %Y"
)

base_longa["ano"] = base_longa["data"].dt.year
base_longa["mes"] = base_longa["data"].dt.month
base_longa["producao"] = pd.to_numeric(
    base_longa["producao"], errors="coerce"
)

print("Linhas após transformação:", len(base_longa))
display(base_longa.head())

## 6. DESAFIO — Construir pelo menos três gráficos

Sugestões:

1. produção total ao longo dos meses;
2. produção acumulada das principais marcas;
3. produção média por mês do ano;
4. produção atual versus produção do mês anterior.

Para cada gráfico, escreva qual hipótese ele ajuda a investigar.

In [ ]:
# GRÁFICO 1 — produção mensal
# 1. Agrupe por data.
# 2. Some a produção.
# 3. Construa um gráfico de linha.

print("DESAFIO: produza o gráfico de evolução mensal.")

In [ ]:
# GRÁFICO 2 — marcas
# 1. Agrupe por VP: Production Brand.
# 2. Some a produção.
# 3. Ordene e selecione as dez maiores marcas.

print("DESAFIO: produza o gráfico de produção por marca.")

In [ ]:
# GRÁFICO 3 — sazonalidade
# 1. Agregue a produção por data.
# 2. Identifique o mês de cada observação.
# 3. Calcule a produção média por mês.

print("DESAFIO: produza o gráfico de sazonalidade.")

## 7. DESAFIO — Preparar a série mensal

Crie uma tabela com uma linha por mês e as colunas `data` e `producao`. Depois construa variáveis temporais e históricas.

> Uma média móvel utilizada para prever o mês atual precisa considerar apenas os meses anteriores.

In [ ]:
serie = (
    base_longa
    .groupby("data", as_index=False)["producao"]
    .sum()
    .sort_values("data")
    .reset_index(drop=True)
)

serie["mes"] = serie["data"].dt.month
serie["trimestre"] = serie["data"].dt.quarter

# COMPLETE:
# serie["producao_mes_anterior"] = ...
# serie["media_movel_3_meses"] = ...
# serie["producao_12_meses_atras"] = ...

display(serie.head())

## 8. DESAFIO — Investigar valores extremos

Calcule Q1, Q3, IQR e os limites para sinalizar possíveis extremos da série mensal. Antes de remover qualquer mês, responda: o valor pode representar férias, parada produtiva ou sazonalidade?

In [ ]:
# COMPLETE:
# q1 = ...
# q3 = ...
# iqr = ...
# limite_inferior = ...
# limite_superior = ...
# possiveis_extremos = ...

print("DESAFIO: sinalize os extremos e explique se devem ser removidos.")

## 9. DESAFIO — Separar treino e teste cronologicamente

Reserve os últimos 12 meses para teste. Não use embaralhamento para este problema.

### Checkpoint de segurança
Antes de escrever o código, responda: qual é a data máxima do treino e qual é a data mínima do teste? Se elas estiverem invertidas ou misturadas, a avaliação não simula o futuro.


In [ ]:
# COMPLETE depois de criar as features e remover apenas os NaN gerados pelas defasagens:
# base_modelagem = ...
# treino = ...
# teste = ...
# features = [...]
# X_treino = ...
# y_treino = ...
# X_teste = ...
# y_teste = ...

print("DESAFIO: o teste deve representar meses posteriores ao treino.")

## 10. DESAFIO — Criar um baseline

Use a produção do mês anterior como previsão ingênua. O modelo linear deverá ser comparado com essa referência nos mesmos meses.

### Preveja antes de calcular
Em quais situações repetir o mês anterior pode funcionar bem? Em quais situações pode falhar? Registre uma frase antes de calcular o MAE.


In [ ]:
# COMPLETE:
# previsao_baseline = teste["producao_mes_anterior"]
# mae_baseline = mean_absolute_error(y_teste, previsao_baseline)

print("DESAFIO: calcule o MAE da previsão ingênua.")

## 11. DESAFIO — Treinar uma regressão linear

Crie um pipeline com imputação, padronização e regressão linear. Ajuste somente com o treino e produza previsões para o teste.

In [ ]:
# COMPLETE:
# modelo = Pipeline(steps=[
#     ("imputar", SimpleImputer(strategy="median")),
#     ("escalar", StandardScaler()),
#     ("regressao", LinearRegression())
# ])
# modelo.fit(..., ...)
# previsoes = modelo.predict(...)

print("DESAFIO: ajuste o modelo e gere previsões.")

## 12. DESAFIO — Calcular e interpretar as métricas

Calcule **MAE**, **MSE** e **R²**. Compare o MAE do modelo com o MAE do baseline. Produza um gráfico com produção observada e prevista.

### Modelo de interpretação
- **MAE:** “em média, o modelo errou aproximadamente ___ veículos por mês”.
- **MSE:** compare com o baseline; valores grandes indicam penalização de erros extremos.
- **R²:** indique se o modelo superou a referência baseada na média no período de teste.

Não transforme MAE em porcentagem sem calcular explicitamente uma métrica percentual.


In [ ]:
# COMPLETE:
# mae = mean_absolute_error(...)
# mse = mean_squared_error(...)
# r2 = r2_score(...)

print("DESAFIO: diga qual estratégia apresentou menor erro e o que isso significa.")

## 13. Bônus — Classificação com KNN

Se o alvo fosse uma categoria, utilizaríamos classificação. Experimente o conjunto Iris do scikit-learn, separação treino/teste com `random_state=42`, `StandardScaler`, `KNeighborsClassifier`, acurácia e matriz de confusão.

In [ ]:
# BÔNUS:
# from sklearn.datasets import load_iris
# iris = load_iris(as_frame=True)
# X = iris.data
# y = iris.target
# ...

print("Bônus: compare a tarefa de classificação com a regressão da produção.")

## 14. Fechamento e documentação 4.2

Complete:

> Nossa primeira hipótese foi __________. Ela foi investigada por __________. O modelo utilizou __________. A divisão respeitou __________. O MAE foi __________ e, em comparação com o baseline, __________.

### Referências

SCIKIT-LEARN DEVELOPERS. *Common pitfalls and recommended practices*. Disponível em: https://scikit-learn.org/stable/common_pitfalls.html.

SCIKIT-LEARN DEVELOPERS. *Metrics and scoring: quantifying the quality of predictions*. Disponível em: https://scikit-learn.org/stable/modules/model_evaluation.html.

THE TURING WAY COMMUNITY. *Definitions*. Disponível em: https://book.the-turing-way.org/reproducible-research/overview/overview-definitions/.

## Autoavaliação final

Marque o que seu grupo conseguiu demonstrar:

- [ ] As três hipóteses respeitam o significado da base.
- [ ] Os três gráficos possuem título, eixos e interpretação.
- [ ] Nenhuma feature utiliza o futuro.
- [ ] O teste contém os meses mais recentes.
- [ ] Baseline e modelo foram comparados nos mesmos meses.
- [ ] MAE, MSE e R² foram explicados no contexto.
- [ ] A conclusão registra pelo menos uma limitação.
